# 06 — Análise dos Filtros Wavelet Aprendidos (SelfRegulationSCP1)

Carrega o melhor modelo `learned_wavelet`, extrai os filtros (h,g) — **um par por canal e por nível** — e inspeciona.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, 'config')
from experiment_config import (
    SCP1_CONFIG, DATA_DIR, RESULTS_DIR, SEED,
    WAVELET_CONFIG, LEARNED_WAVELET_CONFIG, DL_TRAINING_CONFIG,
    ML_MODELS_CONFIG, ML_SEARCH_CONFIG, build_param_dist, N_JOBS_QUARTER,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(SEED)
print('Dataset:', SCP1_CONFIG['dataset_name'], '| classes:', SCP1_CONFIG['n_classes'],
      '| seq_len:', SCP1_CONFIG['sequence_length'], '| canais:', SCP1_CONFIG['n_features'])

## 1. Localizar o melhor modelo learned_wavelet salvo

In [ ]:
import json, tensorflow as tf
runs = sorted([p for p in RESULTS_DIR.glob('????-??-??*') if (p/'queue_status.json').exists()], reverse=True)
assert runs, 'Nenhuma run encontrada.'
cands = []
for f in runs[0].rglob('metrics.json'):
    m = json.loads(f.read_text())
    if 'learned_wavelet' in m.get('mode',''):
        cands.append((float(m.get('test_accuracy', 0) or 0), f.parent))
cands.sort(reverse=True)
best_dir = cands[0][1]; print('melhor learned_wavelet:', best_dir.relative_to(RESULTS_DIR))
model = tf.keras.models.load_model(best_dir / 'model.keras', compile=False)

## 2. Inspecionar pesos da camada LWT

In [ ]:
lwt_layers = [l for l in model.layers if 'Wavelet' in type(l).__name__]
print('camadas wavelet:', [type(l).__name__ for l in lwt_layers])
for l in lwt_layers:
    for w in l.weights:
        print(f'{w.name:40s} {tuple(w.shape)}')

## 3. Próximos passos

Para análise completa dos filtros (QMF, ortogonalidade, momentos de anulação, resposta em
frequência) por **cada um dos 6 canais EEG**, reaproveite as funções do notebook 06 do `ford-a`.

In [ ]:
print('Modelo carregado — pronto para análise espectral por canal.')
print('n_canais =', SCP1_CONFIG['n_features'], '| niveis LWT =', LEARNED_WAVELET_CONFIG['levels'])